In [1]:
import os
from dotenv import load_dotenv
from deepagents import create_deep_agent
from openai import OpenAI

load_dotenv()

# ---- LangSmith tracing ----
langsmith_api_key = os.getenv("LANGSMITH_API_KEY")
if not langsmith_api_key:
    raise ValueError("LANGSMITH_API_KEY is missing. Add it to .env, then rerun this cell.")

os.environ["LANGSMITH_API_KEY"] = langsmith_api_key
os.environ["LANGSMITH_TRACING"] = os.getenv("LANGSMITH_TRACING", "true")
os.environ["LANGSMITH_PROJECT"] = os.getenv("LANGSMITH_PROJECT", "physio")

# Optional: set only if present in .env
if os.getenv("LANGSMITH_ENDPOINT"):
    os.environ["LANGSMITH_ENDPOINT"] = os.getenv("LANGSMITH_ENDPOINT")
if os.getenv("LANGSMITH_WORKSPACE_ID"):
    os.environ["LANGSMITH_WORKSPACE_ID"] = os.getenv("LANGSMITH_WORKSPACE_ID")

# You said you'll use your OpenAI client/key
openai_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

def internet_search(query: str, max_results: int = 5) -> str:
    """Doctor tool: web search using OpenAI web search."""
    response = openai_client.responses.create(
        model="gpt-4o-mini",
        tools=[{"type": "web_search_preview"}],
        tool_choice={"type": "web_search_preview"},
        input=f"Return up to {max_results} concise, reliable results for: {query}",
    )
    return response.output_text

def receptionist_faq_lookup(user_query: str) -> str:
    """Receptionist tool (placeholder): static FAQ response for now."""
    q = user_query.lower()
    if "appointment" in q or "book" in q:
        return "You can book by sending your preferred day/time; clinic hours are 9 AM–6 PM, Sun–Thu."
    if "price" in q or "cost" in q:
        return "Initial assessment is 500 EGP and follow-up sessions are 350 EGP (sample static pricing)."
    if "location" in q or "address" in q:
        return "Main branch: Nasr City, Cairo (static placeholder)."
    return "Thanks for your question. A receptionist will help with appointments, pricing, location, and clinic FAQs."

doctor_system_prompt = """You are the Doctor subagent for a physiotherapy chatbot.
Use internet_search when clinical context requires current best-practice information.
Keep responses clear, safe, and practical.
Do not provide a definitive diagnosis.
If red flags are present, advise urgent in-person medical care."""

receptionist_system_prompt = """You are the Receptionist subagent for a physiotherapy chatbot.
Handle non-clinical FAQs: appointments, pricing, branch/location, clinic timings, and admin questions.
Use receptionist_faq_lookup for answers.
If the user asks a clinical question, tell the main agent to delegate to doctor."""

doctor_skills_source = os.path.abspath("src/app/notebooks/deepagents/skills")

subagents = [
    {
        "name": "doctor",
        "description": "Use for physiotherapy symptoms, pain, injury, rehab, exercises, and red-flag triage.",
        "system_prompt": doctor_system_prompt,
        "tools": [internet_search],
        "skills": [doctor_skills_source],
        "model": "openai:gpt-4o-mini",
    },
    {
        "name": "receptionist",
        "description": "Use for administrative FAQs such as booking, pricing, hours, and location.",
        "system_prompt": receptionist_system_prompt,
        "tools": [receptionist_faq_lookup],
        "model": "openai:gpt-4o-mini",
    },
]

main_router_prompt = """You are the main physiotherapy chatbot router.
Route each user question to exactly one subagent:
- doctor: for clinical or symptom-related questions
- receptionist: for admin/FAQ questions
If mixed, prioritize safety/clinical routing to doctor first."""

agent = create_deep_agent(
    model="openai:gpt-4o-mini",
    system_prompt=main_router_prompt,
    subagents=subagents,
 )

# Quick examples (these runs will appear in LangSmith when LANGSMITH_API_KEY is set)
clinical = agent.invoke(
    {"messages": [{"role": "user", "content": "I have knee pain while climbing stairs for 2 weeks, what should I do?"}]},
    config={"run_name": "physio-clinical-demo", "tags": ["physio", "doctor-route"]},
)
print("Clinical route:\n", clinical["messages"][-1].content)

faq = agent.invoke(
    {"messages": [{"role": "user", "content": "What are your working hours and how can I book an appointment?"}]},
    config={"run_name": "physio-faq-demo", "tags": ["physio", "receptionist-route"]},
)
print("\nFAQ route:\n", faq["messages"][-1].content)

Clinical route:
 [{'type': 'text', 'text': 'If you have knee pain while climbing stairs for the past two weeks, consider the following recommendations for assessment and care:\n\n### Potential Causes\n1. **Patellar Tendonitis:** Inflammation of the tendon connecting the kneecap to the shinbone, often due to overuse.\n2. **Iliotibial Band Syndrome:** Tightness or inflammation of the band along the outer thigh.\n3. **Chondromalacia Patella:** Softening of cartilage beneath the kneecap, causing pain during stair climbing.\n4. **Meniscus Tear:** Tear in the cartilage cushioning the knee joint.\n5. **Osteoarthritis:** Wear and tear of the knee joint due to aging.\n6. **Bursitis:** Inflammation of fluid-filled sacs (bursae) in the knee.\n\n### Recommendations\n#### Initial Care\n- **Rest:** Avoid activities that worsen the pain.\n- **Ice:** Apply to the knee for 15-20 minutes multiple times daily.\n- **Compression:** Use an elastic bandage or knee sleeve to reduce swelling.\n- **Elevation:**